# Hands-on Exercise: Structured Invoice Extraction

Fill in the `TODO(...)` blanks to complete the notebook.

In [ ]:
def TODO(todo: str = "Fill the blank"):
    raise ValueError(todo)

In [ ]:
import os
from openai import OpenAI
import pydantic
import random
from PIL import Image

from ie_course.image import encode_image

In [ ]:
Image.open("../../data/gemini_generated_invoice.png")

In [ ]:
img_bytes = encode_image("../../data/gemini_generated_invoice.png")

### Exercise 1
Load the API key from the environment or just type it here.

In [ ]:
# os.environ["CHATAI_API_KEY"]
key = os.environ[TODO("Environment variable name")]
url = "https://chat-ai.academiccloud.de/v1"

### Exercise 2
Create the API client.

In [ ]:
client = OpenAI(base_url=url, api_key=TODO("API key variable"))

### Exercise 3
Choose a model.

In [ ]:
model = TODO("Model name")

### Exercise 4
Insert the image data URL.

In [ ]:
messages = [
    {
        "role": "system",
        "content": """
        Du extrahierst Informationen wie Rechnungspositionen und Rechnungsbeträge aus gescannten deutschsprachigen 
        Rechnungen und Gutschriften.
        """
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Extract ONLY the total amount due from this invoice. Return only the number."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{TODO("insert img_bytes")}"
                }
            }
        ]
    }
]


In [ ]:
invoice_schema = {
    "type": "object",
    "properties": {
        "vendor_name":    {"type": "string"},
        "buyer_name":    {"type": "string"},
        "invoice_number": {"type": "string"},
        "issue_date":     {"type": "string", "description": "ISO 8601 date, e.g. 2024-09-15"},
        "currency":       {"type": "string", "description": "3-letter ISO code"},
        "status": {
            "type": "string",
            "enum": ["paid", "unpaid", "partially_paid"]
        },
        "line_items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "quantity":    {"type": "integer"},
                    "unit_price":  {"type": "number"}
                },
                "required": ["description", "quantity", "unit_price"],
                "additionalProperties": False
            }
        },
        "total_amount":   {"type": "number"},
        "purchase_order": {"type": ["string", "null"]}
    },
    "required": [
        "vendor_name", "buyer_name", "invoice_number", "issue_date", "currency",
        "status", "line_items", "total_amount", "purchase_order"
    ],
    "additionalProperties": False
}

### Exercise 5
Enable JSON schema output.

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0,
    response_format={
        "type": TODO("Response format type"),
        "json_schema": {
            "name": "ExtractionSchema",
            "schema": invoice_schema,
            "strict": True, 
        }
    },
)

In [ ]:
print(response.choices[0].message.content)